# 11: Kiểm tra trên nhóm test (D19)

**Chỉ chạy một lần, sau khi khoá cấu hình.** Nhóm test (60 user, D17) chưa được dùng để chọn bất kỳ ngưỡng nào. Notebook này chạy bước 7 cho nhóm test, rồi đo **đúng những thước đo đã dùng trên dev** và đặt hai nhóm cạnh nhau.

| Mục | Đo gì | So với dev |
|---|---|---|
| T1 | Cấu hình đang khoá (in từ code) | ghi ngày khoá vào D19 |
| T2 | Có home / office, các cờ, confidence | bảng 31a, 31b (notebook 10) |
| T3 | Nửa đầu / nửa sau có ra cùng nơi theo mức confidence | bảng 30 (notebook 10) |
| T4 | Stay nhận nhầm theo nhãn phương tiện | bảng 4 (notebook 07) |

**Quy tắc (D19):** không sửa code hay ngưỡng sau khi xem số ở đây. Lệch nhiều so với dev thì ghi là hạn chế. Nếu vẫn sửa thì số test này không còn là số test.

**Hạn chế:** đo độ ổn định trên user mới, không đo độ đúng vì không có nhãn home / office (A13).

In [1]:
import importlib
from pathlib import Path
import sys
import numpy as np
import pandas as pd
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
from preprocess import interim
from stays import detect
from places import cluster_v1
from places import home_office_conf as hoc
from eda import split, stay_scan as ss, stay_anchor as sa, conf_holdout as ch
for m in (hoc, ch):
    importlib.reload(m)

dev, test = split.dev_users(), split.test_users()
print(f"dev {len(dev)} user, test {len(test)} user, trùng nhau: {len(set(dev) & set(test))}")

dev 122 user, test 60 user, trùng nhau: 0


## T1. Cấu hình đang khoá

In thẳng từ code để chắc chắn test chạy đúng cấu hình đã chốt. Nếu khác D19 thì dừng lại.

In [2]:
T1 = pd.Series({
    "neo stay": detect.METHOD,
    "D (m)": detect.DIST_M,
    "T (phút)": detect.TIME_S / 60,
    "max gap (giờ)": detect.MAX_GAP_S / 3600,
    "gom địa điểm (m)": cluster_v1.RADIUS_M,
    "khung đêm (giờ)": hoc.NIGHT,
    "khung làm việc (giờ)": hoc.WORK,
    "phút tối thiểu mỗi phiếu": hoc.MIN_VOTE_MIN,
    "mốc phiếu (đêm / ngày làm việc)": (hoc.N_MIN_NIGHTS, hoc.N_MIN_WORKDAYS),
    "số lần bốc lại": hoc.N_BOOT,
    "hạt giống": hoc.SEED,
}, name="giá trị")
display(T1.to_frame())

,giá trị
neo stay,h1_mean
D (m),200.0
T (phút),30.0
max gap (giờ),24.0
gom địa điểm (m),200.0
khung đêm (giờ),"(20.0, 31.0)"
khung làm việc (giờ),"[(8.5, 18.0)]"
phút tối thiểu mỗi phiếu,60.0
mốc phiếu (đêm / ngày làm việc),"(7, 7)"
số lần bốc lại,200


## T2. Kết quả bước 7 trên nhóm test

Chạy `home_office_conf` cho 60 user test và ghi `reports/home_office_v1_test.csv` (giống lệnh `python -m src.places.home_office_conf --test`). **Nếu file đã có thì chỉ đọc lại, không chạy đè**, để số test là của lần chạy đầu tiên. Muốn chạy lại phải xoá file bằng tay và ghi lý do vào D19.

In [3]:
TEST_CSV = ROOT / "reports" / "home_office_v1_test.csv"
DEV_CSV = ROOT / "reports" / "home_office_v1_dev.csv"

if TEST_CSV.exists():
    print(f"Đã có {TEST_CSV.name}: chỉ đọc lại, không chạy đè.")
else:
    TEST_CSV.parent.mkdir(parents=True, exist_ok=True)
    hoc.run(test).to_csv(TEST_CSV, index=False)
    print(f"Đã chạy và ghi {TEST_CSV.name}")
ho = {"dev": pd.read_csv(DEV_CSV, dtype={"user": str}), "test": pd.read_csv(TEST_CSV, dtype={"user": str})}

def tong_quan(d):
    n = len(d)
    f = d["flags"].fillna("")
    pct = lambda m: round(100 * m.sum() / n, 1)
    return {"số user": n, "có home (%)": pct(d["home"].notna()), "có office (%)": pct(d["office"].notna()),
            "có cả hai (%)": pct(d["home"].notna() & d["office"].notna()),
            "home = office (%)": pct(f.str.contains("home_eq_office")),
            "không cờ nào (%)": pct(f == "")}

print("Bảng T2a. Tổng quan")
display(pd.DataFrame({k: tong_quan(v) for k, v in ho.items()}))
print("Bảng T2b. % user theo cờ")
display(pd.DataFrame({k: (100 * v["flags"].fillna("").str.get_dummies(";").sum() / len(v)).round(1) for k, v in ho.items()}).fillna(0))
print("Bảng T2c. Confidence (%)")
display(pd.concat({k: hoc.conf_summary(v) for k, v in ho.items()}, axis=1))

Đã chạy và ghi home_office_v1_test.csv
Bảng T2a. Tổng quan


,dev,test
số user,122.0,60.0
có home (%),84.4,96.7
có office (%),50.8,55.0
có cả hai (%),46.7,55.0
home = office (%),25.4,28.3
không cờ nào (%),23.0,21.7


Bảng T2b. % user theo cờ


,dev,test
few_nights,45.9,46.7
few_workdays,50.0,58.3
home_eq_office,25.4,28.3
no_night,9.0,3.3
no_stays,6.6,0.0
no_workday,17.2,16.7
unstable_home,6.6,5.0
unstable_office,2.5,5.0


Bảng T2c. Confidence (%)


dev         test       
                            home office  home office
có kết quả                   103     62    58     33
có confidence (đủ 7 phiếu)    58     40    32     15
trung vị (%)                96.8   92.7  92.0  100.0
dưới 70%                      16     11     9      4
70–90%                         9      6     5      3
từ 90%                        33     23    18      8

**Đọc kết quả:** nếu tỉ lệ có home / office, các cờ và trung vị confidence của test gần dev (chênh vài điểm % với 60 user là bình thường) thì phương pháp không chỉ khớp riêng nhóm dev. Chênh lớn thì xem có phải do test có nhiều user ít dữ liệu hơn không (cờ `few_*`).

## T3. Nửa đầu / nửa sau theo mức confidence

Giống mục 30 của notebook 10: chia stay của từng user làm hai nửa theo giờ đến, bỏ phiếu riêng, hỏi user có confidence nửa đầu cao thì nửa sau có ra cùng nơi nhiều hơn không. Dev chạy lại tại đây để hai bảng cùng một code.

In [4]:
R = {"dev": ch.run(dev), "test": ch.run(test)}
for kind in ("home", "office"):
    out = {}
    for k, r in R.items():
        g, rho, n = ch.summary(r, kind)
        out[k] = g[["user có confidence nửa đầu", "trùng nơi (% trong số có cả hai)"]].round(1)
        print(f"{kind} · {k}: {n} user có kết quả ở cả hai nửa, tương quan hạng {rho:.2f}")
    print(f"Bảng T3 ({kind})")
    display(pd.concat(out, axis=1))

home · dev: 43 user có kết quả ở cả hai nửa, tương quan hạng 0.32
home · test: 22 user có kết quả ở cả hai nửa, tương quan hạng 0.19
Bảng T3 (home)


dev  \
                   user có confidence nửa đầu   
confidence nửa đầu                              
dưới 70%                                   16   
70–90%                                      2   
từ 90%                                     25   

                                                     \
                   trùng nơi (% trong số có cả hai)   
confidence nửa đầu                                    
dưới 70%                                       25.0   
70–90%                                         50.0   
từ 90%                                         68.0   

                                         test                                   
                   user có confidence nửa đầu trùng nơi (% trong số có cả hai)  
confidence nửa đầu                                                              
dưới 70%                                    6                             33.3  
70–90%                                      7                             57.1  
từ 90%                                      9                             66.7

office · dev: 20 user có kết quả ở cả hai nửa, tương quan hạng 0.34
office · test: 9 user có kết quả ở cả hai nửa, tương quan hạng 0.00
Bảng T3 (office)


dev  \
                   user có confidence nửa đầu   
confidence nửa đầu                              
dưới 70%                                    8   
70–90%                                      2   
từ 90%                                     15   

                                                     \
                   trùng nơi (% trong số có cả hai)   
confidence nửa đầu                                    
dưới 70%                                       50.0   
70–90%                                        100.0   
từ 90%                                         78.6   

                                         test                                   
                   user có confidence nửa đầu trùng nơi (% trong số có cả hai)  
confidence nửa đầu                                                              
dưới 70%                                    3                             50.0  
70–90%                                      2                            100.0  
từ 90%                                      7                             60.0

**Đọc kết quả:** trên dev, mức từ 90% trùng nhiều hơn mức dưới 70% (home 68% so với 25%), một phần lệch là do chuyển nhà thật. Test cần cho cùng chiều. Mẫu test nhỏ hơn nên chỉ đọc chiều, không đọc chênh vài điểm.

## T4. Stay nhận nhầm theo nhãn phương tiện

Giống bảng 4 của notebook 07: % thời gian di chuyển có nhãn bị nhận là stay, trên các user test có `labels.txt`.

In [5]:
XE = {"bus", "car", "taxi", "train", "subway", "airplane", "boat", "motorcycle"}
lab_all = set(interim.list_stems(interim.INTERIM_DIR / "labels")) & set(interim.list_stems(Path(ss.CLEAN_DIR)))
lab = {"dev": sorted(lab_all & set(dev)), "test": sorted(lab_all & set(test))}

def bang_nham(users):
    F = sa.run_labels(users, sa.CMP, verbose=False)
    F = F[F["variant"] == "h1_mean"].copy()
    F["nhóm"] = np.where(F["mode"] == "walk", "walk", np.where(F["mode"].isin(XE), "xe", "khác"))
    g = F.groupby("nhóm")[["hours", "false_hours"]].sum()
    out = {"số user có nhãn": len(users), "tổng (%)": 100 * F["false_hours"].sum() / F["hours"].sum()}
    out.update({f"  {k} (%)": 100 * g.at[k, "false_hours"] / g.at[k, "hours"] for k in ("walk", "xe", "khác") if k in g.index})
    return pd.Series(out)

print("Bảng T4. % thời gian di chuyển bị nhận là stay (neo h1_mean)")
display(pd.DataFrame({k: bang_nham(v) for k, v in lab.items()}).round(2))

Bảng T4. % thời gian di chuyển bị nhận là stay (neo h1_mean)


,dev,test
số user có nhãn,47.00,22.00
tổng (%),10.94,11.76
walk (%),29.89,23.99
xe (%),2.45,3.12
khác (%),6.03,7.02


**Đọc kết quả:** dev là 10,94% (notebook 07). Test gần mức đó thì ngưỡng stay không khớp riêng dev.

## Kết luận (chạy một lần, 2026-11-06)

- **T1:** cấu hình đúng D19 (h1_mean, 200 m, 30 phút, max gap 24 giờ, gom 200 m, đêm 20–7h, làm việc 8:30–18:00, 60 phút, 7 phiếu, 200 lần bốc lại).
- **T2:** test gần dev. Có home 96,7% (dev 84,4%, test không có user nào `no_stays`), có office 55,0% (50,8%), home = office 28,3% (25,4%), `few_nights` 46,7% (45,9%), `few_workdays` 58,3% (50,0%). Trung vị confidence home 92,0% (96,8%), office 100% (92,7%, chỉ 15 user).
- **T3:** home vẫn đúng chiều: trùng nơi 33% / 57% / 67% ở mức dưới 70% / 70–90% / từ 90% (6 / 7 / 9 user), tương quan hạng 0,19 (dev 0,32). Office chỉ 9 user, không đọc được.
- **T4:** stay nhận nhầm 11,76% trên 22 user test có nhãn (dev 10,94%, +0,8 điểm); walk 24,0% (29,9%), xe 3,1% (2,5%).
- **Đọc chung:** các thước đo trên test nằm gần dev, nên ngưỡng không khớp riêng nhóm dev. Mẫu test nhỏ nên chỉ đọc chiều. Không sửa gì sau khi xem số này (D19).